# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The purpose of this playbook is simple: rank content pages so that a human reviewer knows which pages to look at first.

The baseline score uses three simple signals:

- **Staleness:** how long it has been since the page was last updated.
- **Visibility:** how many impressions the page received in the last 90 days.
- **Search opportunity:** the page's search volume.

The score gives more points to pages that are older and still have meaningful search visibility.

The score is only a prioritization signal. It does not mean that a page definitely needs to be changed.

### Baseline scoring rule

The baseline assigns points using simple thresholds:

| Signal | Condition | Points |
|---|---|---:|
| Staleness | `days_since_last_update >= 180` | +3 |
| Visibility | `impressions_90d >= 500` | +2 |
| Search opportunity | `search_volume >= 500` | +1 |

The maximum score is 6.

A score of 3 or more receives the action `REFRESH_REVIEW`. Lower-scoring pages receive `MONITOR`.

This rule is intentionally simple so that a reviewer can understand why a page was ranked highly without needing to inspect a machine-learning model.

### Reason codes

A reason code explains why a page received a particular recommendation.

| Reason code | Simple meaning | Action |
|---|---|---|
| `STALE_VISIBLE_HIGH_OPPORTUNITY` | The page is old, still receives meaningful impressions, and has high search volume. | `REFRESH_REVIEW` |
| `LOW_PRIORITY` | The page does not reach the baseline threshold for refresh review. | `MONITOR` |

The reason code explains the rule-based priority. It does not claim that refreshing the page will cause better search performance.

In [82]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().parents[1]
data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [84]:
# Baseline signals

stale = (df["days_since_last_update"] >= 180).astype(int)

visible = (df["impressions_90d"] >= 500).astype(int)

high_opportunity = (
    df["search_volume"] >= 500
).fillna(False).astype(int)

# Baseline score
df["baseline_score"] = (
    stale * 3 +
    visible * 2 +
    high_opportunity * 1
)

df[
    [
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "search_volume",
        "baseline_score"
    ]
].head(10)

,content_id,days_since_last_update,impressions_90d,search_volume,baseline_score
0,content_304f48230142,20,3803,10.0,2
1,content_a1fb4e703a9e,25,15320,90.0,2
2,content_9aa793d4d895,20,12581,0.0,2
3,content_331d6c4de07b,22,11751,10.0,2
4,content_d99b7a2d90ca,14,19140,0.0,2
5,content_d4084a4bc775,20,3970,720.0,3
6,content_9a34b442b552,20,20,0.0,0
7,content_a63219c6e95a,22,1724,590.0,3
8,content_5e6c160719bc,20,32574,0.0,2
9,content_c27558df2b0c,104,1240,0.0,2


In [98]:
def assign_reason(row):
    score = row["baseline_score"]

    if score == 6:
        return "STALE_VISIBLE_HIGH_OPPORTUNITY"
    elif score == 5:
        return "STALE_VISIBLE"
    elif score == 4:
        return "STALE_HIGH_OPPORTUNITY"
    elif score == 3:
        return "REVIEW_THRESHOLD"
    else:
        return "LOW_PRIORITY"

df["reason_code"] = df.apply(assign_reason, axis=1)

df[
    [
        "content_id",
        "baseline_score",
        "reason_code"
    ]
].head(10)

,content_id,baseline_score,reason_code
0,content_304f48230142,2,LOW_PRIORITY
1,content_a1fb4e703a9e,2,LOW_PRIORITY
2,content_9aa793d4d895,2,LOW_PRIORITY
3,content_331d6c4de07b,2,LOW_PRIORITY
4,content_d99b7a2d90ca,2,LOW_PRIORITY
5,content_d4084a4bc775,3,REVIEW_THRESHOLD
6,content_9a34b442b552,0,LOW_PRIORITY
7,content_a63219c6e95a,3,REVIEW_THRESHOLD
8,content_5e6c160719bc,2,LOW_PRIORITY
9,content_c27558df2b0c,2,LOW_PRIORITY


In [99]:
def assign_action(row):
    if row["baseline_score"] >= 3:
        return "REFRESH_REVIEW"
    else:
        return "MONITOR"

df["action_label"] = df.apply(assign_action, axis=1)

df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].head(10)

,content_id,baseline_score,reason_code,action_label
0,content_304f48230142,2,LOW_PRIORITY,MONITOR
1,content_a1fb4e703a9e,2,LOW_PRIORITY,MONITOR
2,content_9aa793d4d895,2,LOW_PRIORITY,MONITOR
3,content_331d6c4de07b,2,LOW_PRIORITY,MONITOR
4,content_d99b7a2d90ca,2,LOW_PRIORITY,MONITOR
5,content_d4084a4bc775,3,REVIEW_THRESHOLD,REFRESH_REVIEW
6,content_9a34b442b552,0,LOW_PRIORITY,MONITOR
7,content_a63219c6e95a,3,REVIEW_THRESHOLD,REFRESH_REVIEW
8,content_5e6c160719bc,2,LOW_PRIORITY,MONITOR
9,content_c27558df2b0c,2,LOW_PRIORITY,MONITOR


In [100]:
ranked_df = (
    df.sort_values(
        by="baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked_df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].head(20)

,content_id,baseline_score,reason_code,action_label
0,content_b16bd7307b39,5,STALE_VISIBLE,REFRESH_REVIEW
1,content_72496874f806,5,STALE_VISIBLE,REFRESH_REVIEW
2,content_fe16a55cd13d,5,STALE_VISIBLE,REFRESH_REVIEW
3,content_1bfaa38ff26c,5,STALE_VISIBLE,REFRESH_REVIEW
4,content_074ba6ead17b,5,STALE_VISIBLE,REFRESH_REVIEW
5,content_7f116ae1f6f5,5,STALE_VISIBLE,REFRESH_REVIEW
6,content_ecb6215e79fd,5,STALE_VISIBLE,REFRESH_REVIEW
7,content_77d4d5930e5e,5,STALE_VISIBLE,REFRESH_REVIEW
8,content_bdbec75c1148,5,STALE_VISIBLE,REFRESH_REVIEW
9,content_e3ff1b093148,5,STALE_VISIBLE,REFRESH_REVIEW


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended to help content reviewers prioritize pages for human review.

The baseline score ranks pages using observable signals such as page age, impressions, and search volume. A higher score means the page receives higher review priority under this rule.

The queue is designed to answer:

> "Which pages should we look at first?"

It is not designed to automatically decide what content should be changed.

### What the output means

A `REFRESH_REVIEW` action means that the page is worth looking at based on the baseline signals.

It does not mean that the page is definitely declining, that the content is poor, or that refreshing it will improve search performance.

`MONITOR` means that the page does not currently meet the baseline's review threshold.

### Limits

This playbook has several important limits:

- The baseline is a simple prioritization rule, not a causal model.
- A high score does not prove that a page will decline or that refreshing it will increase traffic.
- The queue does not establish anything about Google's ranking algorithm.
- The recommendation should be reviewed by a human before any content change is made.
- The starter dataset provides a snapshot of content and trailing-90-day metrics, so this playbook does not by itself establish future-time performance.
- Results should be treated as decision-support evidence rather than as an automatic content optimization system.

### Decision flow

The intended workflow is:

1. Rank pages using the baseline score.
2. Review the highest-priority pages first.
3. Check the page and its context manually.
4. Decide whether a content change is actually appropriate.
5. Monitor the outcome after any approved change.

The model or rule prioritizes the review; a human makes the final decision.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review before action

Every page in the review queue must be checked by a person before any content change is made.

The reviewer should check:

1. **Is the page actually stale?**
   Confirm that the page needs an update rather than assuming age alone means the content is outdated.

2. **Is the page still relevant to its intended audience?**
   Check whether the topic, search intent, and purpose of the page are still appropriate.

3. **Is there a real content problem?**
   Look for outdated information, missing sections, weak explanations, broken references, or other observable issues.

4. **Could the signals have another explanation?**
   Consider seasonality, campaigns, tracking changes, or other unusual events before taking action.

5. **Is the proposed action appropriate?**
   A high baseline score means "review this page first", not "rewrite this page".

The reviewer can accept, reject, or defer the recommendation.

### No-go list: what should never be automated

The following are not automated by this playbook:
- Automatically rewriting or deleting a page because it received a high score.
- Automatically claiming that a page is declining because it is old.
- Automatically changing content without checking search intent and page context.
- Automatically treating a large performance change as a real content problem without checking for tracking, campaign, or seasonal effects.
- Automatically claiming that a refresh will improve traffic, rankings, or search performance.
- Automatically treating the baseline score as proof of a Google ranking factor.
- Automatically overriding a human reviewer's decision.

The queue is a prioritization and decision-support tool. It raises pages for review; it does not make the final content decision.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.